# Модель 2, шаг 1 — дообучение bi-encoder (InfoNCE), голова поверх замороженного e5

Схема: **e5 заморожен** (веса трансформера не обучаются) + поверх него
обучается только маленькая голова `DoubleGeGLUHead` (два GeGLU-блока с
residual + LayerNorm). Так как e5 заморожен и детерминирован, его выход не
меняется от эпохи к эпохе - незачем гонять тяжёлый forward/backward
трансформера на каждом шаге. Вместо этого:

1. Считаем эмбеддинги e5 (`backbone_only`, без головы) для ВСЕХ запросов и
   объявлений **один раз** и сохраняем на диск (`.npy`).
2. Загружаем их через `np.load(..., mmap_mode="r")` - файлы не грузятся
   целиком в RAM, читаются с диска по требованию (страницами), это и дёшево
   по памяти, и быстро для случайного доступа по батчам.
3. Обучаем голову (AdamW + линейный warmup, контрастивный InfoNCE-лосс с
   in-batch негативами - то же самое, что `MultipleNegativesRankingLoss`,
   но реализовано вручную, т.к. голова работает уже НЕ с текстом, а с
   готовыми векторами) - это на порядки быстрее, потому что e5 в цикле
   обучения больше не участвует вообще.
4. Чекпоиним голову после каждой эпохи (`./head_checkpoints/epoch_N`),
   считаем val recall@50 и в конце собираем финальную модель из ЛУЧШЕЙ
   эпохи.

**Зачем вообще дообучать, а не взять предобученную модель как есть:**
предобученная multilingual-e5 знает общую семантическую близость текстов, а
нам нужна специфичная для Авито-услуг связь "запрос → объявление, которое
реально выбрали" (например "автозвук" → "Услуги автоэлектрика" — без
дообучения модель вряд ли сама выучит такую связку).

Базовая модель по умолчанию — `intfloat/multilingual-e5-base`.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

from text_prep import build_embedding_query_text, build_item_text_for_embedding
from custom_head import DoubleGeGLUHead

## Конфиг — поправьте под себя

In [ ]:
BASE = "."  # папка с train_queries.parquet/train_items.parquet/train_pairs.parquet
#BASE_MODEL = "intfloat/multilingual-e5-small"
BASE_MODEL = "intfloat/multilingual-e5-base"
OUTPUT_DIR = "./biencoder_finetuned"           # сюда сохраняется ФИНАЛЬНАЯ модель (e5 + лучшая голова)
SUMMARIES_PATH = None  # например "./item_summaries.parquet" (item_id, summary) - см. summarize_with_llm.py

N_TRAIN_PAIRS = None   # None = использовать все train_pairs; поставьте число (напр. 50000) для быстрого прогона
VAL_FRACTION = 0.03    # доля запросов, отложенная для оценки (Recall@k) - НЕ участвует в обучении

BATCH_SIZE = 256        # батч ГОЛОВЫ поверх готовых эмбеддингов - e5 в этом батче уже не участвует
                        # (заморожен + закэширован), поэтому можно смело держать большим
EPOCHS = 10
HEAD_LR = 2e-4          # голова маленькая и обучается с нуля - lr побольше, чем для файнтюна всей модели
WARMUP_RATIO = 0.1
SEED = 42

EMBED_BATCH_SIZE_QUERY = 128   # батч для ОДНОРАЗОВОГО прогона e5 по запросам - тексты короткие
EMBED_BATCH_SIZE_ITEM = 32     # батч по объявлениям - тексты длиннее (заголовок+параметры+описание),
                                # меньший батч, чтобы не вылетать в shared memory на 6GB VRAM
EMBED_CACHE_DIR = "./embed_cache"          # .npy файлы с закэшированными эмбеддингами e5 (mmap)
CHECKPOINT_DIR = "./head_checkpoints"      # чекпоинт головы на каждую эпоху

# Явно выбираем устройство: "cuda", если PyTorch видит GPU, иначе "cpu".
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def is_e5_model(name: str) -> bool:
    return "e5" in name.lower()


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Проверяем, что PyTorch видит GPU

In [ ]:
if DEVICE == "cuda":
    print(f"устройство: cuda ({torch.cuda.get_device_name(0)})")
else:
    print(
        "устройство: cpu - torch.cuda.is_available() вернул False.\n"
        "Если у вас есть GPU (nvidia-smi это показывает), значит установленный "
        "PyTorch собран БЕЗ поддержки CUDA (частая ситуация на Windows при "
        "обычном `pip install torch`).\n"
        "Проверьте: torch.__version__ =", torch.__version__, ", torch.version.cuda =", torch.version.cuda, "\n"
        "Если torch.version.cuda is None - переустановите torch с CUDA-сборкой "
        "с https://pytorch.org/get-started/locally/ (версию CUDA возьмите из "
        "nvidia-smi, обычно подходит cu121/cu124/cu128)."
    )

## Загружаем данные

In [ ]:
tq = pd.read_parquet(f"{BASE}/train_queries.parquet")
ti = pd.read_parquet(f"{BASE}/train_items.parquet")
tp = pd.read_parquet(f"{BASE}/train_pairs.parquet")
print(f"queries={len(tq)} items={len(ti)} pairs={len(tp)}")

In [ ]:
summaries = None
if SUMMARIES_PATH and os.path.exists(SUMMARIES_PATH):
    summ_df = pd.read_parquet(SUMMARIES_PATH)  # ожидается колонки item_id, summary
    summaries = summ_df.set_index("item_id")["summary"]
    print(f"loaded {len(summaries)} LLM-суммаризаций объявлений")

## Тексты для эмбеддинг-модели (без BM25-специфичных повторов слов - см. text_prep.py)

In [ ]:
q_text = build_embedding_query_text(tq).set_axis(tq["query_uid"])
item_summaries_aligned = summaries.reindex(ti["item_id"]).reset_index(drop=True) if summaries is not None else None

# ВАЖНО: текст объявления берётся из ОДНОЙ функции с настройками из config_v2
# (EMB_TEXT_MODE / EMB_ITEM_TEXT_KW) - та же функция вызывается в 02, иначе
# голова обучается на одном формате текста, а применяется к другому
# (в прошлой версии 01 резал до 150/200/200, а 02 кодировал с 150/500/500).
it_text = build_item_text_for_embedding(ti, use_summary=item_summaries_aligned).set_axis(ti["item_id"])

e5 = is_e5_model(BASE_MODEL)
q_prefix = "query: " if e5 else ""
p_prefix = "passage: " if e5 else ""
print(f"is_e5_model={e5}  q_prefix={q_prefix!r}  p_prefix={p_prefix!r}")
print(it_text.iloc[0][:300])
q_text.head()

## Train/val split — по `query_uid`, чтобы один и тот же запрос не оказался и в train, и в val

In [ ]:
uniq_uids = tq["query_uid"].unique()
rng = np.random.default_rng(SEED)
rng.shuffle(uniq_uids)
n_val = max(1, int(len(uniq_uids) * VAL_FRACTION))
val_uids = set(uniq_uids[:n_val])
train_uids = set(uniq_uids[n_val:])

pairs = tp.copy()
if N_TRAIN_PAIRS is not None:
    pairs = pairs.sample(n=min(N_TRAIN_PAIRS, len(pairs)), random_state=SEED)

train_pairs = pairs[pairs["query_uid"].isin(train_uids)]
val_pairs = pairs[pairs["query_uid"].isin(val_uids)]
print(f"train pairs: {len(train_pairs)}, val pairs (для оценки): {len(val_pairs)}")

## Позиции пар в tq/ti

Вместо `InputExample`-объектов с текстом (как раньше) держим только ПОЗИЦИИ
строк в `tq`/`ti` - это же будут позиции в кэше эмбеддингов (кэш строится
ниже строго в том же порядке, что и строки `tq`/`ti`), так что обучающий
цикл сможет доставать нужные векторы прямым индексированием по mmap-массиву,
без текста вообще.

In [ ]:
uid_to_pos = {uid: i for i, uid in enumerate(tq["query_uid"].values)}
item_to_pos = {iid: i for i, iid in enumerate(ti["item_id"].values)}
tq_uid_arr = tq["query_uid"].values

train_q_pos = train_pairs["query_uid"].map(uid_to_pos).to_numpy()
train_i_pos = train_pairs["item_id"].map(item_to_pos).to_numpy()
val_q_pos = val_pairs["query_uid"].map(uid_to_pos).to_numpy()
val_i_pos = val_pairs["item_id"].map(item_to_pos).to_numpy()

# на случай "битых" пар (id, которого нет в tq/ti) - такое не ожидается, но на
# всякий случай отфильтруем, а не упадём с ошибкой
train_mask = ~(pd.isna(train_q_pos) | pd.isna(train_i_pos))
train_q_pos, train_i_pos = train_q_pos[train_mask].astype(np.int64), train_i_pos[train_mask].astype(np.int64)
val_mask = ~(pd.isna(val_q_pos) | pd.isna(val_i_pos))
val_q_pos, val_i_pos = val_q_pos[val_mask].astype(np.int64), val_i_pos[val_mask].astype(np.int64)

print(f"training positions: {len(train_q_pos)}, val positions: {len(val_q_pos)}")

## Базовая модель: e5 (заморожен) + голова DoubleGeGLUHead (обучаемая)

In [ ]:
base_model = SentenceTransformer(BASE_MODEL, device=DEVICE)
word_embedding_model = base_model._first_module()   # e5
pooling_model = base_model[1]                          # пулинг

for param in word_embedding_model.auto_model.parameters():
    param.requires_grad = False

hidden_dim = pooling_model.get_sentence_embedding_dimension()
head = DoubleGeGLUHead(dim=hidden_dim).to(DEVICE)   # внутри GeGLU hidden = dim*4 по умолчанию

model = SentenceTransformer(modules=[word_embedding_model, pooling_model, head], device=DEVICE)
model

## Кэш эмбеддингов e5 (mmap)

Прогоняем ЗАМОРОЖЕННЫЙ `e5 + pooling` (без головы) по ВСЕМ запросам и
объявлениям ровно один раз и сохраняем результат в `.npy`. Дальше грузим их
через `mmap_mode="r"` - массив не читается целиком в RAM, а подгружается с
диска по требуемым индексам, что дёшево по памяти и достаточно быстро для
случайного доступа по батчам при обучении головы.

Если файлы уже посчитаны (перезапуск ноутбука) - шаг пропускается.

In [ ]:
os.makedirs(EMBED_CACHE_DIR, exist_ok=True)
query_emb_path = os.path.join(EMBED_CACHE_DIR, "query_emb.npy")
item_emb_path = os.path.join(EMBED_CACHE_DIR, "item_emb.npy")

# backbone_only - e5 + пулинг, БЕЗ головы: единственное, что реально нужно
# прогонять через трансформер. Раз e5 заморожен и детерминирован, его выход
# не меняется между эпохами/батчами - считаем один раз и переиспользуем,
# вместо того чтобы впустую гонять тяжёлый forward трансформера на каждом шаге.
backbone_only = SentenceTransformer(modules=[word_embedding_model, pooling_model], device=DEVICE)
backbone_only.half()   # fp16 инференс - тут только forward без backward, точность эмбеддингов
                        # не страдает заметно, а VRAM на активации и скорость выигрывают в разы

if os.path.exists(query_emb_path) and os.path.exists(item_emb_path):
    print("кэш эмбеддингов уже посчитан, пропускаем пересчёт")
else:
    q_all_text = (q_prefix + q_text).tolist()
    it_all_text = (p_prefix + it_text).tolist()

    q_emb = backbone_only.encode(
        q_all_text, batch_size=EMBED_BATCH_SIZE_QUERY, show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    np.save(query_emb_path, q_emb)
    del q_emb

    it_emb = backbone_only.encode(
        it_all_text, batch_size=EMBED_BATCH_SIZE_ITEM, show_progress_bar=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    np.save(item_emb_path, it_emb)
    del it_emb

    print(f"кэш эмбеддингов сохранён: {query_emb_path}, {item_emb_path}")

# mmap_mode="r" - файлы НЕ грузятся целиком в RAM, читаются с диска по требованию
query_emb_cache = np.load(query_emb_path, mmap_mode="r")
item_emb_cache = np.load(item_emb_path, mmap_mode="r")
print(f"query_emb_cache: {query_emb_cache.shape}  item_emb_cache: {item_emb_cache.shape}")

del backbone_only
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## Оценка: Recall@50 на отложенных запросах (через кэш + текущую голову)

Без тяжёлого `InformationRetrievalEvaluator` - т.к. эмбеддинги корпуса уже
закэшированы, нужно только прогнать их через (маленькую!) голову и посчитать
косинусные близости - это быстро и делается вручную.

In [ ]:
val_uids_present = val_pairs["query_uid"].unique().tolist()
val_query_pos = np.array([uid_to_pos[u] for u in val_uids_present], dtype=np.int64)

# корпус для оценки: объявления из val_pairs (релевантные) + случайный пул
# остальных train_items как "шум"-дистракторы, чтобы оценка была не совсем
# тривиальной (в реальности корпус - все benchmark_items, тут это лишь прокси)
rel_items_pos = np.unique(val_i_pos)
rng_eval = np.random.default_rng(SEED)
non_rel_pos = np.setdiff1d(np.arange(len(ti)), rel_items_pos)
distractor_pool_pos = rng_eval.choice(
    non_rel_pos, size=min(20000, len(non_rel_pos)), replace=False,
)
eval_corpus_pos = np.unique(np.concatenate([rel_items_pos, distractor_pool_pos]))

relevant_docs = {}
for uid, ipos in zip(val_pairs["query_uid"], val_i_pos):
    relevant_docs.setdefault(uid, set()).add(int(ipos))

print(f"eval queries: {len(val_query_pos)}, eval corpus size: {len(eval_corpus_pos)}")


@torch.no_grad()
def evaluate_recall_at_k(head, k=50, batch_size=512):
    head.eval()
    corpus_vecs = []
    for start in range(0, len(eval_corpus_pos), batch_size):
        idx = eval_corpus_pos[start:start + batch_size]
        x = torch.from_numpy(np.asarray(item_emb_cache[idx], dtype=np.float32)).to(DEVICE)
        out = F.normalize(head({"sentence_embedding": x})["sentence_embedding"], dim=-1)
        corpus_vecs.append(out.cpu())
    corpus_vecs = torch.cat(corpus_vecs, dim=0)  # (n_corpus, dim)

    recalls = []
    for start in range(0, len(val_query_pos), batch_size):
        idx = val_query_pos[start:start + batch_size]
        x = torch.from_numpy(np.asarray(query_emb_cache[idx], dtype=np.float32)).to(DEVICE)
        out = F.normalize(head({"sentence_embedding": x})["sentence_embedding"], dim=-1).cpu()
        sims = out @ corpus_vecs.T
        topk_idx = sims.topk(min(k, sims.shape[1]), dim=-1).indices.numpy()
        for row_i, qpos in enumerate(idx):
            uid = tq_uid_arr[qpos]
            rel = relevant_docs.get(uid)
            if not rel:
                continue
            hit_pos = set(eval_corpus_pos[topk_idx[row_i]].tolist())
            recalls.append(len(hit_pos & rel) / len(rel))
    head.train()
    return float(np.mean(recalls)) if recalls else 0.0


print("recall@50 ДО обучения (голова со случайной инициализацией):", evaluate_recall_at_k(head, k=50))

## Обучение головы поверх кэша (без e5 в цикле)

Ручной цикл: AdamW + линейный warmup, InfoNCE вручную (косинус между
`head(q)` и `head(i)`, умноженный на `SCALE`, кросс-энтропия с диагональными
метками - то же самое, что `MultipleNegativesRankingLoss`, но без текста и
без прогона через трансформер на каждом шаге). Чекпоиним голову после каждой
эпохи и в конце выбираем лучшую по `val recall@50`.

In [ ]:
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

n_train = len(train_q_pos)
steps_per_epoch = (n_train + BATCH_SIZE - 1) // BATCH_SIZE
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

optimizer = AdamW(head.parameters(), lr=HEAD_LR)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

SCALE = 20.0  # тот же default scale, что в MultipleNegativesRankingLoss (cosine * scale перед CE)

history = []
best_epoch, best_recall = -1, -1.0

for epoch in range(EPOCHS):
    head.train()
    perm = np.random.default_rng(SEED + epoch).permutation(n_train)
    epoch_loss = 0.0
    pbar = tqdm(range(steps_per_epoch), desc=f"epoch {epoch + 1}/{EPOCHS}")
    for step in pbar:
        batch_idx = perm[step * BATCH_SIZE:(step + 1) * BATCH_SIZE]
        if len(batch_idx) == 0:
            continue
        q_pos_b = train_q_pos[batch_idx]
        i_pos_b = train_i_pos[batch_idx]

        q_x = torch.from_numpy(np.asarray(query_emb_cache[q_pos_b], dtype=np.float32)).to(DEVICE)
        i_x = torch.from_numpy(np.asarray(item_emb_cache[i_pos_b], dtype=np.float32)).to(DEVICE)

        q_out = F.normalize(head({"sentence_embedding": q_x})["sentence_embedding"], dim=-1)
        i_out = F.normalize(head({"sentence_embedding": i_x})["sentence_embedding"], dim=-1)

        sims = (q_out @ i_out.T) * SCALE   # (batch, batch) - InfoNCE с in-batch негативами
        labels = torch.arange(sims.shape[0], device=DEVICE)
        loss = F.cross_entropy(sims, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = epoch_loss / steps_per_epoch
    recall = evaluate_recall_at_k(head, k=50)
    history.append({"epoch": epoch + 1, "loss": avg_loss, "recall@50": recall})
    print(f"epoch {epoch + 1}/{EPOCHS}  loss={avg_loss:.4f}  val recall@50={recall:.4f}")

    ckpt_path = os.path.join(CHECKPOINT_DIR, f"epoch_{epoch + 1}")
    head.save(ckpt_path)
    if recall > best_recall:
        best_recall, best_epoch = recall, epoch + 1

print(f"\nлучшая эпоха: {best_epoch} (val recall@50={best_recall:.4f})")
pd.DataFrame(history)

## Собираем финальную модель из лучшей эпохи и сохраняем

In [ ]:
best_head = DoubleGeGLUHead.load(os.path.join(CHECKPOINT_DIR, f"epoch_{best_epoch}")).to(DEVICE)

final_model = SentenceTransformer(modules=[word_embedding_model, pooling_model, best_head], device=DEVICE)
final_model.save(OUTPUT_DIR)
print(f"лучшая модель (эпоха {best_epoch}, val recall@50={best_recall:.4f}) сохранена в {OUTPUT_DIR}")

## Дальше

Перейдите в `02_embed_and_retrieve.ipynb` — там `MODEL_PATH` уже указывает
сюда (`./biencoder_finetuned`) по умолчанию, менять ничего не нужно: на
выходе всё равно получается обычная папка с полной моделью (e5 + голова),
она её подхватит как раньше.

Чекпоинты всех эпох остаются в `./head_checkpoints/epoch_N`, если захотите
позже сравнить их между собой или откатиться на другую эпоху.